## **Library**

In [19]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from catboost import CatBoostClassifier
import seaborn as sns

import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

In [20]:
df = pd.read_csv("playground-series-s6e6/train.csv")

baris, kolom = df.shape
print(f"Baris: {baris}, Kolom: {kolom}")
df.head()

Baris: 577347, Kolom: 12


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


## **Data Cleaning**

In [21]:
missing_data = df.isnull().sum()
missing_data = missing_data.sort_values(ascending= False)
missing_data

id                   0
alpha                0
delta                0
u                    0
g                    0
r                    0
i                    0
z                    0
redshift             0
spectral_type        0
galaxy_population    0
class                0
dtype: int64

Data tidak ada yang hilang

In [22]:
y = df["class"]
X = df.drop(["class"], axis=1)
cat_col = ['spectral_type', 'galaxy_population']

## **Mutual Information**

In [23]:
# Label encoding for categoricals
for colname in X.select_dtypes("object"):
    X[colname], _ = X[colname].factorize()
diskrit = X.dtypes == int

In [ ]:
def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

mi_scores = make_mi_scores(X, y, diskrit)
mi_scores

In [ ]:
model = CatBoostClassifier(
    iterations=1000, 
    learning_rate= 0.05,
    depth = 8,
    loss_function= 'MultiClass',
    eval_metric= 'TotalF1',
    cat_features= cat_col,
    random_seed= 42)

## **Feature Engineering**

In [ ]:
#seberapa  dekat ke infrared
df["u-g"] = df["u"] - df["g"]
df["g-r"] = df["g"] - df["r"]
df["r-i"] = df["r"] - df["i"]
df["i-z"] = df["i"] - df["z"]

fe = ["u-g", 'g-r','r-i', 'i-z']

### **MI Score after Feature Engineering**

In [ ]:
y = df["class"]
X = df.drop(["class", "alpha", "delta"], axis=1)

# Label encoding for categoricals
for colname in X.select_dtypes("object"):
    X[colname], _ = X[colname].factorize()
diskrit = X.dtypes == int

def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

mi_scores = make_mi_scores(X, y, diskrit)
mi_scores

id                   0.879847
redshift             0.514990
g-r                  0.321157
spectral_type        0.299984
r-i                  0.218324
z                    0.211517
galaxy_population    0.192690
u                    0.178881
g                    0.174585
u-g                  0.171347
i                    0.158246
r                    0.097940
i-z                  0.074759
Name: MI Scores, dtype: float64

### **Clustering**

In [ ]:
def  Clustering(df, col, n, name, is_scaling=True):
    X_cluster = df[col]
    
    if not is_scaling:
        X_scaled = StandardScaler().fit_transform(X_cluster)
    else:
        X_scaled = X_cluster

    kmeans = KMeans(n_clusters= n, random_state=42)
    cluster = kmeans.fit_predict(X_scaled)
    X_cluster = pd.DataFrame(cluster, columns=[name])
    X_cluster = X_cluster.astype("category")
    
    return X_cluster

In [ ]:
filter_col = ['u-g', 'g-r', 'r-i', 'i-z']

cluster = Clustering(df, filter_col, 3, 'group_colour', is_scaling= True)

cluster.head()


,group_colour
0,2
1,2
2,1
3,2
4,2


### **PCA**

In [ ]:
def PCA(df, col, is_scaling= True):
    pca = PCA()
    X_pca = df[col]

    if is_scaling:
        X_scaled = StandardScaler().fit_transform(X_pca)
    else:
        X_scaled = X_pca

    X_pca = pca.fit_transform(X_scaled)

    component_names = [f"PC{i+1}" for i in range(X_pca.shape[1])]
    X_pca = pd.DataFrame(X_pca, columns=component_names)

    loadings = pd.DataFrame(
    pca.components_.T,  # transpose the matrix of loadings
    columns=component_names,  # so the columns are the principal components
    index=X_pca.columns,  # and the rows are the original features
)
    return X_pca, loadings

In [ ]:
X_pca, loadings = PCA(df, filter_col)

loadings

TypeError: PCA() missing 2 required positional arguments: 'df' and 'col'